In [31]:
import pandas as pd
import getpass
from sqlalchemy import create_engine, text, URL

password = getpass.getpass("Please enter your SQL password: ")

url = URL.create(
    "mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="bookworms"
)

engine = create_engine(url)

# Test the connection.
with engine.connect() as connection:
    rows = connection.execute(text("SHOW TABLES")).fetchall()

print(rows)

Please enter your SQL password:  ········


[('authors',), ('book_authors',), ('books',), ('collection_of_books',), ('collections',)]


In [32]:
df = pd.read_csv("../data/clean/bookworms_cleaned_data_file (1).csv")

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  str    
 2   authors             11118 non-null  str    
 3   average_rating      11097 non-null  float64
 4   isbn                11123 non-null  str    
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  str    
 7   num_pages           11047 non-null  float64
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  str    
 11  publisher           11123 non-null  str    
 12  publication_year    11123 non-null  int64  
dtypes: float64(2), int64(5), str(6)
memory usage: 1.1 MB


# database population
### Load the books. 
Python selects the needed columns from the cleaned CSV and inserts them into books. It keeps the original Goodreads ID as book_id.
### Link books to authors. 
Python splits the author names, saves each distinct name in authors, and uses the generated IDs to add one row per book–author pair to book_authors.
### Build the four collections. 
The code saves each collection’s name and selection rule in collections. It then runs those rules against books and adds each qualifying book–collection pair to collection_of_books. A book can qualify for more than one collection.


In [36]:
#column selection and matching datatypes as mentioned in sql create_databse query

source = df.rename(columns=str.strip)

book_columns = [
    "bookID", "title", "average_rating", "language_code",
    "num_pages", "ratings_count", "text_reviews_count", "publication_year"
]

books = source[book_columns].rename(columns={"bookID": "book_id"}).copy()

if books.empty or books["book_id"].isna().any() or not books["book_id"].is_unique:
    raise ValueError("The CSV must contain books with unique, non-missing book IDs.")

books["num_pages"] = books["num_pages"].astype("Int64")
books["publication_year"] = books["publication_year"].astype("Int64")

new_engine = create_engine(engine.url.set(database="bookworms_new"))

In [37]:
#Python splits the author names, saves each distinct name in authors, and uses the generated IDs to add one row per book–author pair to book_authors

credits = source[["bookID", "authors"]].dropna(subset=["authors"]).rename(
    columns={"bookID": "book_id"}
)
credits["author_name"] = credits["authors"].str.split("/")
credits = credits.explode("author_name")
credits["author_name"] = credits["author_name"].str.strip()
credits = credits.loc[credits["author_name"].ne(""), ["book_id", "author_name"]]
credits = credits.drop_duplicates()

rules = [
    (
        "Popular Favourites",
        "average_rating >= 4.00 AND ratings_count >= 10000",
        "Offer books with broad reader interest."
    ),
    (
        "Recommended Short Reads",
        "average_rating >= 4.00 AND ratings_count >= 500 AND num_pages BETWEEN 3 AND 100",
        "Offer shorter books for readers with limited time."
    ),
    (
        "Hidden-Gem Candidates",
        "average_rating >= 4.50 AND ratings_count BETWEEN 100 AND 999",
        "Help readers discover highly rated, less widely reviewed books."
    ),
    (
        "Popular Classics",
        "average_rating >= 4.00 AND publication_year < 1976",
        "Shortlist older editions for editorial review."
    ),
]

with new_engine.begin() as connection:
    books.to_sql("books", connection, if_exists="append", index=False, chunksize=1000)

    credits[["author_name"]].drop_duplicates().to_sql(
        "authors", connection, if_exists="append", index=False
    )
    authors = pd.read_sql(text("SELECT author_id, author_name FROM authors"), connection)
    author_links = credits.merge(authors, on="author_name", validate="many_to_one")
    author_links[["book_id", "author_id"]].to_sql(
        "book_authors", connection, if_exists="append", index=False, chunksize=1000
    )

    for name, rule, purpose in rules:
        criteria = rule

        result = connection.execute(
            text("""
                INSERT INTO collections
                    (collection_name, selection_criteria, business_purpose)
                VALUES (:name, :criteria, :purpose)
            """),
            {"name": name, "criteria": criteria, "purpose": purpose}
        )

        connection.execute(
            text(f"""
                INSERT INTO collection_of_books (collection_id, book_id)
                SELECT :collection_id, book_id
                FROM books
                WHERE {criteria}
            """),
            {"collection_id": result.lastrowid}
        )

# Qurying the database using python

In [38]:
with new_engine.connect() as connection:
    sql_book_count = connection.execute(text("SELECT COUNT(*) FROM books")).scalar_one()
    print(f"Books in MySQL: {sql_book_count:,} | Rows in CSV: {len(books):,}")

    collection_counts = pd.read_sql(text("""
        SELECT c.collection_name, COUNT(cb.book_id) AS book_count
        FROM collections AS c
        LEFT JOIN collection_of_books AS cb ON cb.collection_id = c.collection_id
        GROUP BY c.collection_id, c.collection_name
        ORDER BY c.collection_id
    """), connection)

display(collection_counts) # result without applying the english language filter used for selection of books

Books in MySQL: 11,123 | Rows in CSV: 11,123


,collection_name,book_count
0,Popular Favourites,976
1,Recommended Short Reads,187
2,Hidden-Gem Candidates,46
3,Popular Classics,105
